In [45]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [46]:
import datetime, os, keras
import pandas as pd
import numpy as np
import seaborn as sns
from keras.utils import to_categorical
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

#from src.models.assemble_model import build_detection_model
from src.miscellaneous import create_seq_dataset_multiple_input_single_output

In [47]:
file_name_list = os.listdir('data/on-road_test_aml')
file_name_list.sort()

In [48]:
file_name_list[:-3]

['data20260122-105722.csv',
 'data20260122-111134.csv',
 'data20260122-144434.csv',
 'data20260122-145111.csv',
 'data20260122-152104.csv']

In [49]:
file_name_list[:len(file_name_list)-3]

['data20260122-105722.csv',
 'data20260122-111134.csv',
 'data20260122-144434.csv',
 'data20260122-145111.csv',
 'data20260122-152104.csv']

In [50]:
train_data_list = []

for file_name in file_name_list[:-3]:
    path = os.path.join('data/on-road_test_aml', file_name)
    data = pd.read_csv(path)
    data = data[data['boom_angle(deg)'] > 0]
    train_data_list.append(data)

train_data = pd.concat(train_data_list, axis=0)
train_data.reset_index(drop=True, inplace=True)

In [51]:
train_data

,time(sec),boom_length(m),boom_angle(deg),load_weight(ton),engine_speed(rpm),wind_speed(m/s),swing_angle(deg),body_angle_x(deg),body_angle_y(deg),load_cell_left_1,load_cell_left_2,load_cell_left_3,load_cell_right_1,load_cell_right_2,load_cell_right_3,load_ratio,roll_over_state,pred,detection
0,0.100,15.047,30.5,3.303,748,-0.598,258,-0.36,0.26,1.15,0.000,1.440,1.540,0.00,1.300,0.90372,0,0.99994,1
1,0.200,15.047,30.5,3.303,748,-0.598,258,-0.36,0.26,1.15,0.000,1.440,1.540,0.00,1.300,0.90372,0,0.99993,1
2,0.301,15.055,30.5,3.303,748,-0.598,258,-0.36,0.26,1.15,0.000,1.440,1.540,0.00,1.300,0.90372,0,0.99984,1
3,0.401,15.055,30.5,3.303,748,-0.598,258,-0.36,0.27,1.14,0.000,1.440,1.540,0.00,1.300,0.90216,0,0.99957,1
4,0.501,15.055,30.5,3.303,748,-0.598,258,-0.36,0.27,1.14,0.000,1.440,1.540,0.00,1.300,0.90216,0,0.98842,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27434,600.700,15.045,29.6,3.875,748,-0.598,258,-0.25,0.25,3.41,20.306,21.074,3.338,3.41,17.674,0.29871,1,0.00000,0
27435,600.800,15.020,29.6,3.875,748,-0.598,258,-0.25,0.25,3.41,20.306,21.074,3.338,3.41,17.674,0.29871,1,0.00000,0
27436,600.900,15.020,29.6,3.875,748,-0.598,258,-0.25,0.25,3.41,20.306,21.074,3.338,3.41,17.674,0.29871,1,0.00000,0
27437,601.000,15.020,29.6,3.875,748,-0.598,258,-0.25,0.25,3.41,20.306,21.074,3.338,3.41,17.674,0.29871,1,0.00000,0


In [52]:
train_boom_angle = train_data['boom_angle(deg)']
train_body_x_axis_angle = train_data['body_angle_x(deg)']
boom_length = 15
train_load_weight = train_data['load_weight(ton)']

ref_working_distance = [4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 9.0, 10.0, 12.0, 14.0, 16.0]
ref_load_capacity = [150, 135, 123.4, 113.5, 97.7, 85.6, 73.4, 62.8, 48.4, 39.1, 14.5]

working_distance = np.cos(np.deg2rad((train_boom_angle-train_body_x_axis_angle))) * boom_length
current_ref_load_capacity = np.interp(working_distance, ref_working_distance, ref_load_capacity)

train_load_usage = (train_load_weight / current_ref_load_capacity) * 100
train_load_usage = train_load_usage.to_numpy()

In [53]:
bins = [0, 8, 15, 100]
labels = [0, 1, 2]
train_target_int = pd.cut(train_load_usage, bins=bins, labels=labels, include_lowest=True)
train_target_one_hot = to_categorical(train_target_int, num_classes=3)

In [54]:
feature_name_list = ['boom_angle(deg)', 'load_weight(ton)', 'engine_speed(rpm)', 'body_angle_x(deg)', 'body_angle_y(deg)']
train_feature = train_data[feature_name_list]
train_feature = train_feature.to_numpy()

In [55]:
train_dataset = np.concatenate([train_feature, train_target_one_hot], axis=1)
train_dataset.shape

(27439, 8)

In [56]:
val_data_list = []

for file_name in file_name_list[:len(file_name_list)-3]:
    path = os.path.join('data/on-road_test_aml', file_name)
    data = pd.read_csv(path)
    data = data[data['boom_angle(deg)'] > 0]
    val_data_list.append(data)

val_data = pd.concat(val_data_list, axis=0)
val_data.reset_index(drop=True, inplace=True)

In [57]:
val_boom_angle = val_data['boom_angle(deg)']
val_body_x_axis_angle = val_data['body_angle_x(deg)']
boom_length = 15
val_load_weight = val_data['load_weight(ton)']

ref_working_distance = [4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 9.0, 10.0, 12.0, 14.0, 16.0]
ref_load_capacity = [150, 135, 123.4, 113.5, 97.7, 85.6, 73.4, 62.8, 48.4, 39.1, 14.5]

working_distance = np.cos(np.deg2rad((val_boom_angle-val_body_x_axis_angle))) * boom_length
current_ref_load_capacity = np.interp(working_distance, ref_working_distance, ref_load_capacity)

val_load_usage = (val_load_weight / current_ref_load_capacity) * 100
val_load_usage = val_load_usage.to_numpy()

In [58]:
bins = [0, 8, 15, 100]
labels = [0, 1, 2]
val_target_int = pd.cut(val_load_usage, bins=bins, labels=labels, include_lowest=True)
val_target_one_hot = to_categorical(val_target_int, num_classes=3)

In [59]:
feature_name_list = ['boom_angle(deg)', 'load_weight(ton)', 'engine_speed(rpm)', 'body_angle_x(deg)', 'body_angle_y(deg)']
val_feature = val_data[feature_name_list]
val_feature = val_feature.to_numpy()

In [60]:
val_dataset = np.concatenate([val_feature, val_target_one_hot], axis=1)
val_dataset.shape

(27439, 8)

In [ ]:
seq_len = 50
pred_distance = 0

train_feature, train_target = create_seq_dataset_multiple_input_single_output(data=train_dataset,
                                                                              seq_len=seq_len,
                                                                              pred_distance=pred_distance,
                                                                              target_idx_pos=train_dataset.shape[1]-3)

train_feature = train_feature.astype(np.float32)
train_target_enc = train_target_enc.astype(np.float32)

print(train_feature.shape, train_target_enc.shape)
print((train_feature.itemsize*train_feature.size)/(1024**2))
print((train_target_enc.itemsize*train_target_enc.size)/(1024**2))